# Entrega 2

Un día de posiciones AIS (NOAA) queda en el volume compartido y, desde ahí, en la tabla
bronze. El día lo marca el widget `fecha`. Esta parte guarda el archivo crudo: no corrige
SOG, coordenadas ni tipos de buque. La unica limpieza es una fila por `(MMSI, BaseDateTime)`.

El catálogo `dbr_acad_ingsistemas_mine` y el esquema `default` son los del curso, y no son
compartidos, cada usuario solo ve los que creo. Cada objeto lleva el prefijo `oceanwatch_`. Este notebook es independiente
del de la Parte 1.

Corre en el editor web de Databricks y en la extensión de VS Code contra el cluster
`1114-011114-9wt67574` (Databricks Connect) {NOTE: solo web, ya no databricks connect}. Hay que tener ese cluster linkeado antes de
ejecutar. En Connect el proceso corre en el laptop, donde el volume no está montado, y la
subida de archivos usa la Files API, como en la Parte 1.

Ejecutar de arriba hacia abajo. Si cambias un widget, vuelve a correr la celda de
configuración: el proceso solo ve el valor cuando esa celda se ejecuta.


## Declarar variables para rutas

In [ ]:
import datetime as dt
import re

# Editor web: spark y dbutils ya existen.
# Extension / Databricks Connect: hay que abrir la sesion en el cluster.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()

try:
    dbutils
except NameError:
    from pyspark.dbutils import DBUtils

    dbutils = DBUtils(spark)

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Mismos nombres que las variables del bundle. El default deja el notebook
# listo en el editor web; un job puede pasar catalog, schema, volume y fecha.
dbutils.widgets.text("catalog", "dbr_acad_ingsistemas_mine", "Catalogo")
dbutils.widgets.text("schema", "default", "Esquema")
dbutils.widgets.text("volume", "oceanwatch_data", "Volume")
dbutils.widgets.text("fecha", "2023-06-08", "Fecha (YYYY-MM-DD)")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()
fecha = dbutils.widgets.get("fecha").strip()

_identificador = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
for _nombre, _valor in (("catalog", catalog), ("schema", schema), ("volume", volume)):
    if not _identificador.match(_valor):
        raise ValueError(f"{_nombre} debe ser un identificador SQL simple. Llego: {_valor!r}")

try:
    fecha_dia = dt.date.fromisoformat(fecha)
except ValueError as exc:
    raise ValueError(f"fecha debe tener la forma YYYY-MM-DD. Llego: {fecha!r}") from exc

# BaseDateTime viene en UTC. El parseo no debe depender de la zona del cluster.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# NOTE: {quita esta funcion. mantiene el codigo directo}
def sql_id(*partes: str) -> str:
    return ".".join(f"`{parte}`" for parte in partes)


VOLUME_NAME = sql_id(catalog, schema, volume)
BRONZE_TABLE = sql_id(catalog, schema, "oceanwatch_bronze_ais")
VOLUME_ROOT = f"/Volumes/{catalog}/{schema}/{volume}"
RAW_ZIPS = f"{VOLUME_ROOT}/raw_zips"
LANDING = f"{VOLUME_ROOT}/landing"
CHECKPOINTS = f"{VOLUME_ROOT}/checkpoints"
SCHEMAS = f"{VOLUME_ROOT}/_schemas"
REF = f"{VOLUME_ROOT}/ref"
STAGING = f"{RAW_ZIPS}/_staging"
CHECKPOINT = f"{CHECKPOINTS}/bronze_ais"
SCHEMA_LOC = f"{SCHEMAS}/bronze_ais"

CARPETAS = [
    ("raw_zips", RAW_ZIPS),
    ("landing", LANDING),
    ("checkpoints", CHECKPOINTS),
    ("_schemas", SCHEMAS),
    ("ref", REF),
]

print(f"Volume: {VOLUME_ROOT}")
print(f"Bronze: {BRONZE_TABLE}")
print(f"Fecha:  {fecha_dia.isoformat()}")


## Configuración, volume y carpetas

Los widgets `catalog`, `schema` y `volume` arrancan con los valores del bundle. `fecha`
es el día a ingerir (`YYYY-MM-DD`), el default `2023-06-08` es el día siguiente a la
semana de la Parte 1. Un job futuro puede pasar esos cuatro nombres como parámetros.

| Carpeta | Para qué |
|---|---|
| `raw_zips/` | Zip del día, como lo publica NOAA |
| `landing/` | CSV ya completos. Única ruta que vigila el stream |
| `checkpoints/` | Checkpoint de la capa bronze |
| `_schemas/` | Esquema que persiste Auto Loader |
| `ref/` | Referencias. El World Port Index entra después |

`CREATE VOLUME IF NOT EXISTS` y las carpetas se pueden repetir.

Dónde se escriben los archivos depende de quién ejecuta Python. `volume_is_local`
comprueba si **este** proceso ve `/Volumes/<catalog>/<schema>/<volume>`. En el driver
(editor web, cluster compartido) esa ruta es el mount del volume: las carpetas se crean
con `os.makedirs` y los archivos se copian con Python. No se usa `dbutils.fs` ni una
ruta `file:` del disco local, porque en modo shared eso exige el privilegio ANY FILE.
En el laptop (Connect) el volume no está montado —en macOS `/Volumes` existe igual, por
los discos— y la subida va por `WorkspaceClient.files`, como en la Parte 1.


In [ ]:
import errno
import os
import shutil
from pathlib import Path

from databricks.sdk.errors import DatabricksError


def volume_is_local(path: str | None = None) -> bool:
    """True si este proceso ve el volume en su filesystem.

    En el editor web el driver monta /Volumes/<catalog>/<schema>/<volume>.
    En Databricks Connect el proceso es el laptop. No basta con mirar /Volumes:
    en macOS ese directorio existe y el catalogo del curso no esta ahi.
    """
    objetivo = VOLUME_ROOT if path is None else path
    return os.path.isdir(objetivo)


def _codigo_error(exc: Exception) -> str:
    return str(getattr(exc, "error_code", "") or "").upper()


def ensure_dir(directory: str, en_driver: bool | None = None) -> None:
    """Crea la carpeta. Se puede llamar varias veces."""
    if en_driver is None:
        en_driver = volume_is_local()
    if en_driver:
        # Cluster compartido: os sobre el mount. dbutils.fs.mkdirs no hace falta
        # y en algunos paths dispara la revision de ANY FILE.
        os.makedirs(directory, exist_ok=True)
        return
    if directory != VOLUME_ROOT and not directory.startswith(VOLUME_ROOT + "/"):
        raise ValueError(f"Ruta fuera del volume: {directory}")
    relativo = directory[len(VOLUME_ROOT) :].strip("/")
    acumulado = VOLUME_ROOT
    for parte in relativo.split("/"):
        if not parte:
            continue
        acumulado = f"{acumulado}/{parte}"
        try:
            w.files.create_directory(acumulado)
        except DatabricksError as exc:
            if _codigo_error(exc) not in {"RESOURCE_ALREADY_EXISTS", "ALREADY_EXISTS"}:
                raise


def volume_file_exists(path: str, en_driver: bool | None = None) -> bool:
    if en_driver is None:
        en_driver = volume_is_local()
    if en_driver:
        return os.path.isfile(path)
    try:
        info = w.files.get_metadata(path)
    except DatabricksError as exc:
        if _codigo_error(exc) == "NOT_FOUND":
            return False
        raise
    return not bool(getattr(info, "is_directory", False))


def copy_local_file_to_volume(local_path: Path, remote_path: str, en_driver: bool | None = None) -> None:
    """Copia un archivo ya completo a una ruta del volume.

    En el driver se escribe con Python sobre /Volumes. En Connect el origen esta
    en el laptop, asi que la subida es la Files API (dbutils.fs.cp file: leeria
    el disco del cluster y, en un cluster compartido, pide ANY FILE).
    """
    if en_driver is None:
        en_driver = volume_is_local()
    parent = remote_path.rsplit("/", 1)[0]
    ensure_dir(parent, en_driver=en_driver)
    if en_driver:
        shutil.copyfile(local_path, remote_path)
        return
    with local_path.open("rb") as handle:
        w.files.upload(remote_path, handle, overwrite=True)


def mover_en_volume(origen: str, destino: str) -> None:
    """Publica `origen` en `destino` dentro del volume.

    En el driver, os.replace en el mismo mount es un rename atomico. Si el FUSE
    no renombra, shutil.move copia y borra. En Connect las dos rutas ya son del
    volume: dbutils.fs.mv no recibe un file: local.
    """
    if not volume_is_local():
        dbutils.fs.mv(origen, destino)
        return
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    try:
        os.replace(origen, destino)
    except OSError as exc:
        no_rename = {
            errno.EXDEV,
            errno.ENOTSUP,
            getattr(errno, "EOPNOTSUPP", errno.ENOTSUP),
            errno.EPERM,
            errno.ENOSYS,
        }
        if exc.errno not in no_rename:
            raise
        print(f"os.replace no disponible en este mount ({exc}). Se usa shutil.move.")
        shutil.move(origen, destino)


In [ ]:
spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}
    COMMENT 'Archivos de OceanWatch AIS: zips NOAA, CSV de landing, checkpoints, esquemas de Auto Loader y referencias.'
    """
)

en_driver = volume_is_local()
print("Carpetas via", "filesystem del driver" if en_driver else "Files API")
for nombre, ruta in CARPETAS:
    ensure_dir(ruta, en_driver=en_driver)
    print(f"Listo {nombre}: {ruta}")


## Descarga automatica del día

URL del widget `fecha`:

`https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{YYYY}/AIS_{YYYY}_{MM}_{DD}.zip`

1. Si `landing/AIS_YYYY_MM_DD.csv` ya pesa lo que dice el zip, no se descarga ni se extrae.
2. Si `raw_zips/AIS_YYYY_MM_DD.zip` pasa `verify_zip_integrity`, no se vuelve a bajar.
   Si falta o está corrupto, `download_with_retries` lo escribe en `raw_zips/_staging/`
   y solo entonces lo pasa a `raw_zips/`.
3. Si el CSV en `raw_zips/_staging/` ya tiene el tamaño del miembro del zip, no se extrae
   otra vez. Si no, se extrae ahí.
4. Ese CSV entra a `landing/` con `os.replace` (mismo volume: rename atómico). Si el mount
   no acepta el rename, se usa `shutil.move`.

En el editor web el proceso es el driver del cluster compartido. Ahí no hay `/tmp` de
staging, ni `dbutils.fs`, ni rutas `file:`: copiar desde el disco local del driver pide
el privilegio ANY FILE (`SELECT on any file`), y esta cuenta no lo tiene. Todo el archivo
se lee y se escribe sobre el mount `/Volumes`.

En Connect el temporal sí está en el laptop. La subida es `WorkspaceClient.files.upload`
y el move final es `dbutils.fs.mv` entre dos rutas del volume.

Para forzar otra descarga hay que borrar el CSV de `landing/` y el zip de `raw_zips/`.
Si el stream ya leyó esa ruta, Auto Loader no la repite (`cloudFiles.allowOverwrites` en
falso). Se puede subir de nuevo con otro nombre. El MERGE de la sección 3 evita que esa
segunda copia duplique filas.


In [ ]:
import datetime as dt
import os
import shutil
import tempfile
import time
import zipfile
from pathlib import Path

import requests


def nombre_csv_dia(day: dt.date) -> str:
    return f"AIS_{day.year}_{day.month:02d}_{day.day:02d}.csv"


def nombre_zip_dia(day: dt.date) -> str:
    return f"AIS_{day.year}_{day.month:02d}_{day.day:02d}.zip"


def url_zip_dia(day: dt.date) -> str:
    return (
        f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{day.year}/"
        f"{nombre_zip_dia(day)}"
    )


def download_with_retries(url: str, dest_path: Path, max_retries: int = 3, delay: int = 5) -> None:
    """Descarga url en dest_path. Reintenta y reemplaza el archivo en cada intento."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Descargando {dest_path.name} (intento {attempt}/{max_retries})...")
            with requests.get(url, stream=True, timeout=(30, 120)) as response:
                response.raise_for_status()
                with dest_path.open("wb") as handle:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                    handle.flush()
                    try:
                        os.fsync(handle.fileno())
                    except OSError:
                        pass
            return
        except Exception as exc:
            print(f"Intento {attempt} fallo: {exc}")
            dest_path.unlink(missing_ok=True)
            if attempt == max_retries:
                raise RuntimeError(
                    f"No se pudo descargar {url} despues de {max_retries} intentos."
                ) from exc
            time.sleep(delay)


def verify_zip_integrity(zip_filepath: Path) -> bool:
    """Verifica que el zip sea valido y no este corrupto."""
    print(f"Verificando integridad de {zip_filepath.name}...")
    if not zip_filepath.is_file() or not zipfile.is_zipfile(zip_filepath):
        return False
    with zipfile.ZipFile(zip_filepath, "r") as zf:
        # testzip devuelve None si no hay archivos danados.
        bad_file = zf.testzip()
        return bad_file is None


def _miembros_csv(zf: zipfile.ZipFile) -> list[str]:
    return [
        nombre
        for nombre in zf.namelist()
        if nombre.lower().endswith(".csv")
        and not nombre.endswith("/")
        and "__MACOSX" not in nombre.split("/")
    ]


def tamano_csv_en_zip(zip_path: Path) -> int:
    """Bytes del unico CSV, segun el directorio central del zip (sin descomprimir)."""
    with zipfile.ZipFile(zip_path) as zf:
        miembros = _miembros_csv(zf)
        if len(miembros) != 1:
            raise ValueError(
                f"Se esperaba un CSV dentro de {zip_path.name}; hay {len(miembros)}: {miembros}"
            )
        return zf.getinfo(miembros[0]).file_size


def extract_single_csv(zip_path: Path, dest_csv: Path) -> None:
    """Copia el unico CSV del zip a dest_csv, con el nombre canonico del dia."""
    dest_csv.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        miembros = _miembros_csv(zf)
        if len(miembros) != 1:
            raise ValueError(
                f"Se esperaba un CSV dentro de {zip_path.name}; hay {len(miembros)}: {miembros}"
            )
        with zf.open(miembros[0]) as origen, dest_csv.open("wb") as destino:
            shutil.copyfileobj(origen, destino, length=8 * 1024 * 1024)
            destino.flush()
            try:
                os.fsync(destino.fileno())
            except OSError:
                pass
    if dest_csv.stat().st_size == 0:
        raise ValueError(f"El CSV extraido quedo vacio: {dest_csv.name}")


def publish_csv(local_csv: Path, remote_final: str, en_driver: bool) -> None:
    """Deja el CSV en landing solo cuando el archivo ya esta completo."""
    staging_path = f"{STAGING}/{local_csv.name}"
    if en_driver:
        ensure_dir(STAGING, en_driver=True)
        if Path(staging_path) != local_csv:
            shutil.copyfile(local_csv, staging_path)
        mover_en_volume(staging_path, remote_final)
        return
    copy_local_file_to_volume(local_csv, staging_path, en_driver=False)
    # Origen y destino son rutas del volume. No hay file: local.
    dbutils.fs.mv(staging_path, remote_final)


def _ingerir_en_driver(day: dt.date) -> str:
    """Descarga, extrae y publica sobre /Volumes. Sin /tmp y sin dbutils.fs.

    Un cluster compartido niega dbutils.fs contra file:/tmp (ANY FILE). El zip y
    el CSV se escriben en raw_zips/_staging y se renombran dentro del mismo volume.
    """
    csv_name = nombre_csv_dia(day)
    zip_name = nombre_zip_dia(day)
    zip_final = Path(RAW_ZIPS) / zip_name
    zip_parcial = Path(STAGING) / f"{zip_name}.partial"
    csv_staging = Path(STAGING) / csv_name
    csv_landing = Path(LANDING) / csv_name

    for carpeta in (RAW_ZIPS, LANDING, STAGING):
        ensure_dir(carpeta, en_driver=True)

    if zip_final.is_file() and verify_zip_integrity(zip_final):
        print(f"Zip valido en raw_zips ({zip_final.stat().st_size} bytes). No se descarga.")
        zip_parcial.unlink(missing_ok=True)
    else:
        if zip_final.exists():
            print("El zip en raw_zips no paso la integridad. Se vuelve a bajar.")
            zip_final.unlink()
        url = url_zip_dia(day)
        print(f"Fuente: {url}")
        download_with_retries(url, zip_parcial)
        if not verify_zip_integrity(zip_parcial):
            zip_parcial.unlink(missing_ok=True)
            raise ValueError(
                f"La verificacion de integridad fallo para {zip_name}. El archivo esta corrupto."
            )
        print(f"Integridad verificada para {zip_name} ({zip_parcial.stat().st_size} bytes).")
        mover_en_volume(str(zip_parcial), str(zip_final))
        print(f"Zip guardado en {zip_final}")

    esperado = tamano_csv_en_zip(zip_final)
    if csv_landing.is_file() and csv_landing.stat().st_size == esperado:
        print(
            f"El CSV de {day.isoformat()} ya esta en landing ({esperado} bytes). No se extrae."
        )
        print(csv_landing)
        csv_staging.unlink(missing_ok=True)
        return str(csv_landing)

    if csv_staging.is_file() and csv_staging.stat().st_size == esperado:
        print(f"CSV en staging ya completo ({esperado} bytes). No se extrae de nuevo.")
    else:
        extract_single_csv(zip_final, csv_staging)
        print(f"CSV extraido: {csv_staging.name} ({csv_staging.stat().st_size} bytes).")
        if csv_staging.stat().st_size != esperado:
            raise ValueError(
                f"El CSV extraido mide {csv_staging.stat().st_size} bytes; el zip declara {esperado}."
            )

    if csv_landing.exists():
        csv_landing.unlink()
    mover_en_volume(str(csv_staging), str(csv_landing))
    print(f"CSV publicado en {csv_landing}")
    return str(csv_landing)


def _ingerir_por_api(day: dt.date) -> str:
    """Connect: el temporal esta en el laptop. Al volume se entra por la Files API."""
    remote_csv = f"{LANDING}/{nombre_csv_dia(day)}"
    if volume_file_exists(remote_csv, en_driver=False):
        print(f"El CSV de {day.isoformat()} ya esta en landing. No se descarga de nuevo.")
        print(remote_csv)
        return remote_csv

    url = url_zip_dia(day)
    zip_name = nombre_zip_dia(day)
    print(f"Fuente: {url}")
    with tempfile.TemporaryDirectory() as tmp:
        tmp_dir = Path(tmp)
        zip_path = tmp_dir / zip_name
        download_with_retries(url, zip_path)
        if not verify_zip_integrity(zip_path):
            raise ValueError(
                f"La verificacion de integridad fallo para {zip_name}. El archivo esta corrupto."
            )
        print(f"Integridad verificada para {zip_name} ({zip_path.stat().st_size} bytes).")
        local_csv = tmp_dir / nombre_csv_dia(day)
        extract_single_csv(zip_path, local_csv)
        print(f"CSV extraido: {local_csv.name} ({local_csv.stat().st_size} bytes).")
        copy_local_file_to_volume(zip_path, f"{RAW_ZIPS}/{zip_name}", en_driver=False)
        print(f"Zip guardado en {RAW_ZIPS}/{zip_name}")
        publish_csv(local_csv, remote_csv, en_driver=False)

    print(f"CSV publicado en {remote_csv}")
    return remote_csv


def ingest_day(day: dt.date) -> str:
    en_driver = volume_is_local()
    if en_driver:
        print("Modo de escritura: Python sobre /Volumes")
        return _ingerir_en_driver(day)
    print("Modo de escritura: Files API (Databricks Connect)")
    return _ingerir_por_api(day)


In [ ]:
# Idempotente: si el CSV de fecha_dia ya esta en landing, no vuelve a bajar el zip.
ingest_day(fecha_dia)


## Stream bronze

Auto Loader lee `landing/` con el esquema explícito de la Parte 1.
Se agregan `_source_file` (desde `_metadata.file_path`) y `_ingested_at`. El checkpoint
queda en `checkpoints/bronze_ais` y el esquema de Auto Loader en `_schemas/bronze_ais`.
`trigger(availableNow=True)` procesa lo que haya en landing y termina, lo cual sirve para
subir un archivo manualmente o desde un job.

La tabla se crea vacia antes del stream. El `MERGE` necesita que exista.
Si una ejecucion anterior dejó `oceanwatch_bronze_ais` con otras columnas,
`CREATE TABLE IF NOT EXISTS` no la modifica, entonces hay que borrarla a mano y volver a crearla.

### Duplicados por buque y timestamp

Cada microbatch hace `dropDuplicates(["MMSI", "BaseDateTime"])` y un `MERGE` con una sola
acción: `WHEN NOT MATCHED THEN INSERT`. Una fila ya escrita no se actualiza ni se borra.
Bronze sigue siendo la copia cruda de la primera vez que vimos esa clave.

Esta vía aguanta un archivo reentregado mejor que `withWatermark` sobre `BaseDateTime`
más `dropDuplicatesWithinWatermark`. El watermark solo recuerda claves dentro de una
ventana de event-time. Pasada la ventana, el mismo mensaje se inserta otra vez. Además
`foreachBatch` es at-least-once: si el proceso se cae después de escribir y antes de
comprometer el offset, Spark repite el lote. Con el `MERGE`, la tabla es la lista de
claves y el reintento queda en cero filas nuevas. Cubre también un CSV con otro nombre,
un checkpoint que se reconstruye y los archivos que siguen en `landing/`.

Dentro del lote, `dropDuplicates` se queda con una fila cualquiera por clave. Si el mismo
`(MMSI, BaseDateTime)` trae coordenadas distintas (el caso de posiciones incompatibles de
la Parte 1), bronze conserva la primera que entró. Silver puede revisarlo.

En el `ON` del `MERGE`, un `NULL` no iguala a otro `NULL`. Un mensaje sin `MMSI` o sin
`BaseDateTime` no queda atado a la clave. Dentro del lote, `dropDuplicates` sí los
colapsa en una fila. El checkpoint sigue impidiendo releer el mismo archivo.

**El stream no se puede correr en dos sesiones a la vez.** El checkpoint es uno solo, en
el volume compartido. Dos queries sobre ese directorio lo corrompen. `landing/` y la tabla
bronze también son compartidas.


In [ ]:
from pyspark.sql import functions as F

# Mismo esquema explicito de la Parte 1 (src/proyecto_ocean_watch.ipynb).
AIS_COLUMNS = [
    ("MMSI", "string"),
    ("BaseDateTime", "timestamp"),
    ("LAT", "double"),
    ("LON", "double"),
    ("SOG", "double"),
    ("COG", "double"),
    ("Heading", "double"),
    ("VesselName", "string"),
    ("IMO", "string"),
    ("CallSign", "string"),
    ("VesselType", "int"),
    ("Status", "int"),
    ("Length", "double"),
    ("Width", "double"),
    ("Draft", "double"),
    ("Cargo", "string"),
    ("TransceiverClass", "string"),
]
ais_schema = ",\n".join(f"    {nombre} {tipo}" for nombre, tipo in AIS_COLUMNS)
nombres_ais = [nombre for nombre, _tipo in AIS_COLUMNS]

SQL_TYPE = {
    "string": "STRING",
    "timestamp": "TIMESTAMP",
    "double": "DOUBLE",
    "int": "INT",
}
TABLE_COMMENT = (
    "Capa bronze de OceanWatch AIS: mensajes crudos e inmutables. "
    "Una fila por (MMSI, BaseDateTime), un reenvio no inserta otra copia. "
    "_source_file y _ingested_at son metadatos de la ingesta."
)
comentario_sql = TABLE_COMMENT.replace("'", "''")
columnas_sql = ",\n".join(f"  {nombre} {SQL_TYPE[tipo]}" for nombre, tipo in AIS_COLUMNS)
columnas_sql += """
  , _source_file STRING COMMENT 'Ruta del CSV en landing (_metadata.file_path).'
  , _ingested_at TIMESTAMP COMMENT 'Instante en que el microbatch escribio la fila.'
"""

# NOTE: {just write the sql code directly. no need to use the variables above, it makes it harder to read}
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    {columnas_sql}
    )
    USING DELTA
    COMMENT '{comentario_sql}'
    """
)
# NOTE: why do we need to comment on the table again? it was already commented in the create table statement
spark.sql(f"COMMENT ON TABLE {BRONZE_TABLE} IS '{comentario_sql}'")

filas_comment = (
    spark.sql(f"DESCRIBE TABLE EXTENDED {BRONZE_TABLE}")
    .where(F.lower(F.col("col_name")) == "comment")
    .select("data_type")
    .collect()
)
if filas_comment:
    print("Comentario en la tabla:", filas_comment[0]["data_type"])
else:
    print("COMMENT aplicado. DESCRIBE EXTENDED no mostro la fila Comment en esta sesion.")


In [ ]:
from pyspark.sql import functions as F


def leer_landing():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .option("cloudFiles.schemaEvolutionMode", "none")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.allowOverwrites", "false")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .schema(ais_schema)
        .load(LANDING)
        .select(
            *[F.col(nombre) for nombre in nombres_ais], # NOTE: just write the columns, remove the list comprehension
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
    )


def merge_into_bronze(batch_df, batch_id: int) -> None:
    """Inserta claves nuevas. Una fila ya presente no se actualiza.

    Corre en el driver del cluster. Solo cierra sobre BRONZE_TABLE (un string)
    para que foreachBatch pueda serializarla en Databricks Connect.
    """
    sesion = batch_df.sparkSession
    lote = batch_df.dropDuplicates(["MMSI", "BaseDateTime"])
    vista = f"oceanwatch_bronze_batch_{int(batch_id)}"
    lote.createOrReplaceTempView(vista)
    sesion.sql(
        f"""
        MERGE INTO {BRONZE_TABLE} AS destino
        USING {vista} AS origen
        ON destino.MMSI = origen.MMSI
       AND destino.BaseDateTime = origen.BaseDateTime
        WHEN NOT MATCHED THEN INSERT *
        """
    )
    sesion.catalog.dropTempView(vista)


def detener_consulta_previa() -> None:
    for consulta in list(spark.streams.active):
        if consulta.name == "oceanwatch_bronze_ais":
            print(
                "Consulta bronze activa en esta sesion. "
                "Se detiene y se retoma desde el checkpoint."
            )
            consulta.stop()


def filas_bronze() -> int:
    fila = spark.sql(f"SELECT COUNT(*) AS n FROM {BRONZE_TABLE}").collect()[0]
    return int(fila["n"])


def filas_leidas(query) -> int:
    eventos = list(query.recentProgress or [])
    if not eventos and getattr(query, "lastProgress", None):
        eventos = [query.lastProgress]
    total = 0
    for evento in eventos:
        if isinstance(evento, dict):
            n = int(evento.get("numInputRows") or 0)
            batch = evento.get("batchId")
        else:
            n = int(getattr(evento, "numInputRows", 0) or 0)
            batch = getattr(evento, "batchId", None)
        total += n
        print(f"batch {batch}: numInputRows={n}")
    return total


def run_bronze_stream():
    print(f"Checkpoint compartido: {CHECKPOINT}")
    print("No ejecutes este stream en paralelo con otra sesion.")
    detener_consulta_previa()
    consulta = (
        leer_landing()
        .writeStream.foreachBatch(merge_into_bronze)
        .queryName("oceanwatch_bronze_ais")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .start()
    )
    consulta.awaitTermination()
    return consulta


antes = filas_bronze()
consulta = run_bronze_stream()
leidas = filas_leidas(consulta)
despues = filas_bronze()
print(f"Corrida: leidas={leidas} antes={antes} despues={despues}")
if leidas == 0 and antes == 0:
    print("No entro ningun archivo. Revisa que landing tenga el CSV del dia.")


## Verificación

Conteo de filas por fecha en UTC, una muestra con las columnas
de la ingesta, y cuántas claves `(MMSI, BaseDateTime)` quedaron repetidas.

In [ ]:
from pyspark.sql import functions as F

total = filas_bronze()
print(f"Filas en bronze: {total}")

por_fecha = (
    spark.table(BRONZE_TABLE)
    .groupBy(F.to_date(F.col("BaseDateTime")).alias("fecha"))
    .agg(F.count("*").alias("filas"))
    .orderBy("fecha")
)
display(por_fecha)

display(
    spark.table(BRONZE_TABLE)
    .select("MMSI", "BaseDateTime", "LAT", "LON", "_source_file", "_ingested_at")
    .limit(5)
)

repetidas = spark.sql(
    f"""
    SELECT COUNT(*) AS claves_repetidas
    FROM (
      SELECT MMSI, BaseDateTime
      FROM {BRONZE_TABLE}
      WHERE MMSI IS NOT NULL AND BaseDateTime IS NOT NULL
      GROUP BY MMSI, BaseDateTime
      HAVING COUNT(*) > 1
    ) repetidas
    """
)
display(repetidas)
n_repetidas = int(repetidas.collect()[0]["claves_repetidas"])
print(f"Claves (MMSI, BaseDateTime) repetidas: {n_repetidas}")

if total == 0:
    raise AssertionError(
        "Bronze tiene 0 filas. Revisa la descarga y que el stream haya leido landing."
    )
if n_repetidas != 0:
    raise AssertionError(
        f"Hay {n_repetidas} claves repetidas. El MERGE debio dejar una fila por "
        "(MMSI, BaseDateTime)."
    )


La siguiente celda vuelve a lanzar el stream. Con el CSV ya procesado, `numInputRows`
tiene que ser 0 y el conteo de la tabla igual porque el checkpoint ya marcó el archivo.

In [ ]:
antes = filas_bronze()
consulta = run_bronze_stream()
leidas = filas_leidas(consulta)
despues = filas_bronze()
print(f"Segunda corrida: leidas={leidas} antes={antes} despues={despues}")
if leidas != 0 or despues != antes:
    raise AssertionError(
        "Se esperaba la segunda corrida con 0 filas nuevas "
        f"(leidas={leidas}, antes={antes}, despues={despues}). "
        "Si el stream todavia no habia corrido, ejecuta antes la celda que lo arranca."
    )
print("Checkpoint: el segundo pase no ingreso filas.")


## Tolerancia a fallos

En primer lugar, si la celda del stream se interrumpe, volver a ejecutarla retoma desde el checkpoint.
`run_bronze_stream` detiene primero una consulta con el mismo nombre que haya quedado
activa en esta sesión, y Auto Loader no vuelve a leer archivos ya comprometidos. Un batch
que alcanzó a hacer `MERGE` y luego se repite no duplica filas: la clave ya está en la tabla.

Ahora, en el caso en en el que se suba un archivo con datos ya procesados y con distinto nombre, estos datos no son duplicados. Para probarlo, la celda de abajo copia el CSV del día a otro nombre dentro de `landing/` y borra la copia
al terminar. En el driver la copia es `shutil` sobre `/Volumes`. En Connect es
`dbutils.fs.cp` entre dos rutas del volume, sin tocar el disco local. El checkpoint no
conoce ese nombre, así que el archivo se lee. El conteo de la tabla no debe cambiar. El
nombre lleva la hora para que repetir la celda no repita el nombre del archivo que el checkpoint
ya vio.


In [ ]:
origen = f"{LANDING}/{nombre_csv_dia(fecha_dia)}"
if not volume_file_exists(origen):
    raise FileNotFoundError(
        f"No esta el CSV del dia en landing ({origen}). Corre antes la descarga."
    )

marca = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
copia = f"{LANDING}/redrop-{marca}-{nombre_csv_dia(fecha_dia)}"
print(f"Copia de reenvio (en el cluster): {copia}")
# En el driver, shutil sobre /Volumes. dbutils.fs.cp de un file: local pide ANY FILE.
if volume_is_local():
    shutil.copyfile(origen, copia)
else:
    dbutils.fs.cp(origen, copia)
try:
    antes = filas_bronze()
    consulta = run_bronze_stream()
    leidas = filas_leidas(consulta)
    despues = filas_bronze()
    print(f"Reenvio: leidas={leidas} antes={antes} despues={despues}")
    if despues != antes:
        raise AssertionError(
            f"El reenvio inserto filas (antes={antes}, despues={despues}). "
            "El MERGE por (MMSI, BaseDateTime) debio dejar la tabla igual."
        )
    if leidas <= 0:
        raise AssertionError(
            "El stream no leyo la copia. Este paso no alcanzo a probar el MERGE."
        )
    print("Reenvio sin filas nuevas. La clave ya estaba en bronze.")
finally:
    if volume_is_local():
        if os.path.isfile(copia):
            os.remove(copia)
    else:
        dbutils.fs.rm(copia)
    print(f"Copia eliminada: {copia}")
